# 🎯 Amaliy mashg'ulot: Korrelyatsiya va Regressiya

## 📚 Maqsad
Ushbu amaliy mashg'ulotda siz o'rgangan nazariy bilimlarni turli ma'lumotlar to'plamida qo'llaysiz.

## 📋 Vazifalar ro'yxati
1. **Sotuv ma'lumotlari tahlili**
2. **Korrelyatsiya tahlili**
3. **Regressiya modeli yaratish**
4. **Model sifatini baholash**
5. **Bashorat qilish**

In [ ]:
# Kutubxonalarni yuklash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## 📊 1. Ma'lumotlarni yuklash va o'rganish

In [ ]:
# Sotuv ma'lumotlarini yuklash
sales_data = pd.read_csv('datasets/sales_data.csv')

print("📊 SOTUV MA'LUMOTLARI:")
print("=" * 25)
print(f"Ma'lumotlar o'lchami: {sales_data.shape}")
print(f"\nUstunlar: {list(sales_data.columns)}")
print(f"\nBirinchi 5 qator:")
print(sales_data.head())

print(f"\n📈 Asosiy statistikalar:")
print(sales_data.describe())

### 🤔 Savol 1: Ma'lumotlarni tahlil qiling
1. Qaysi o'zgaruvchilar raqamli?
2. Qaysi o'zgaruvchi bog'liq o'zgaruvchi (target) bo'lishi mumkin?
3. Ma'lumotlarda yo'qolgan qiymatlar bormi?

In [ ]:
# JAVOB: Bu yerda javob yozing

# Yo'qolgan qiymatlarni tekshirish
print("❌ Yo'qolgan qiymatlar:")
print(sales_data.isnull().sum())

# O'zgaruvchilar turlari
print(f"\n📊 O'zgaruvchilar turlari:")
print(sales_data.dtypes)

## 📊 2. Korrelyatsiya tahlili

### Vazifa 2.1: Korrelyatsiya matritsasini yaratish

In [ ]:
# Korrelyatsiya matritsasini hisoblash va vizualizatsiya qilish
plt.figure(figsize=(10, 8))

# VAZIFA: Bu yerda korrelyatsiya matritsasini yarating
# Maslahat: sales_data.corr() va sns.heatmap() dan foydalaning

# Sizning kodingiz:
corr_matrix = sales_data.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', cbar_kws={'label': 'Korrelyatsiya'})
plt.title('Korrelyatsiya matritsasi')
plt.tight_layout()
plt.show()

# Sotuv hajmi bilan eng kuchli korrelyatsiya
sales_corrs = corr_matrix['sales_volume'].drop('sales_volume').sort_values(key=abs, ascending=False)
print("\n📊 SOTUV HAJMI BILAN KORRELYATSIYA:")
print("=" * 35)
for feature, corr_val in sales_corrs.items():
    print(f"   {feature:20s}: {corr_val:6.3f}")

### Vazifa 2.2: Eng kuchli bog'lanishlarni vizualizatsiya qilish

In [ ]:
# Eng kuchli korrelyatsiyaga ega xususiyatlarni aniqlash
top_features = sales_corrs.head(3).index.tolist()
print(f"📈 Eng kuchli bog'lanishlar: {top_features}")

# Scatter plotlar yaratish
plt.figure(figsize=(15, 5))

for i, feature in enumerate(top_features):
    plt.subplot(1, 3, i+1)
    
    # VAZIFA: Scatter plot yarating
    # x o'qi: feature
    # y o'qi: sales_volume
    
    # Sizning kodingiz:
    plt.scatter(sales_data[feature], sales_data['sales_volume'], alpha=0.7)
    
    # Korrelyatsiya koeffitsiyentini hisoblash
    corr_val = sales_data[feature].corr(sales_data['sales_volume'])
    
    plt.xlabel(feature.replace('_', ' ').title())
    plt.ylabel('Sotuv hajmi')
    plt.title(f'{feature}\nr = {corr_val:.3f}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Vazifa 2.3: Pearson vs Spearman korrelyatsiya

In [ ]:
# Reklama byudjeti va sotuv hajmi uchun ikki xil korrelyatsiya
feature = 'advertising_budget'
target = 'sales_volume'

# VAZIFA: Pearson va Spearman korrelyatsiyasini hisoblang
# Maslahat: stats.pearsonr() va stats.spearmanr() dan foydalaning

# Sizning kodingiz:
pearson_corr, pearson_p = stats.pearsonr(sales_data[feature], sales_data[target])
spearman_corr, spearman_p = stats.spearmanr(sales_data[feature], sales_data[target])

print(f"📊 {feature.upper()} VA {target.upper()} KORRELYATSIYA:")
print("=" * 50)
print(f"   🔢 Pearson korrelyatsiya:  {pearson_corr:.3f} (p = {pearson_p:.3f})")
print(f"   📈 Spearman korrelyatsiya: {spearman_corr:.3f} (p = {spearman_p:.3f})")
print(f"   📊 Farq: {abs(pearson_corr - spearman_corr):.3f}")

# Vizualizatsiya
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(sales_data[feature], sales_data[target], alpha=0.7)
plt.xlabel(feature.replace('_', ' ').title())
plt.ylabel(target.replace('_', ' ').title())
plt.title(f'Chiziqli bog\'lanish\nPearson: {pearson_corr:.3f}')
plt.grid(True, alpha=0.3)

# Regressiya chizig'i qo'shish
z = np.polyfit(sales_data[feature], sales_data[target], 1)
p = np.poly1d(z)
plt.plot(sales_data[feature], p(sales_data[feature]), "r--", alpha=0.8, linewidth=2)

plt.subplot(1, 2, 2)
# Ranglar bo'yicha scatter plot
x_ranks = stats.rankdata(sales_data[feature])
y_ranks = stats.rankdata(sales_data[target])
plt.scatter(x_ranks, y_ranks, alpha=0.7, color='orange')
plt.xlabel(f'{feature.replace("_", " ").title()} (ranglar)')
plt.ylabel(f'{target.replace("_", " ").title()} (ranglar)')
plt.title(f'Rang bog\'lanishi\nSpearman: {spearman_corr:.3f}')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📈 3. Regressiya modeli yaratish

### Vazifa 3.1: Oddiy chiziqli regressiya

In [ ]:
# Eng kuchli bog'lanishga ega xususiyat bilan regressiya modeli
best_feature = sales_corrs.index[0]  # Eng kuchli korrelyatsiya
print(f"📈 Eng yaxshi xususiyat: {best_feature}")

# VAZIFA: Regressiya modeli yarating
# X: best_feature
# y: sales_volume

# Sizning kodingiz:
X = sales_data[[best_feature]]
y = sales_data['sales_volume']

# Model yaratish va o'qitish
model = LinearRegression()
model.fit(X, y)

# Model parametrlari
slope = model.coef_[0]
intercept = model.intercept_

print(f"\n📊 MODEL PARAMETRLARI:")
print("=" * 25)
print(f"   🔢 Og'ish koeffitsiyenti (a): {slope:.3f}")
print(f"   📍 Kesishma nuqtasi (b): {intercept:.3f}")
print(f"\n📝 Regressiya tenglamasi:")
print(f"   Sotuv hajmi = {slope:.3f} × {best_feature} + {intercept:.3f}")

### Vazifa 3.2: Model sifatini baholash

In [ ]:
# Bashorat qilish
y_pred = model.predict(X)

# VAZIFA: Model sifatini baholang
# R², RMSE, MAE ko'rsatkichlarini hisoblang

# Sizning kodingiz:
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = np.mean(np.abs(y - y_pred))

print(f"📊 MODEL SIFATI:")
print("=" * 20)
print(f"   🎯 R² (aniqlash koeffitsiyenti): {r2:.3f} ({r2*100:.1f}%)")
print(f"   📏 RMSE (ildiz o'rtacha kvadrat xatolik): {rmse:.2f}")
print(f"   📊 MAE (o'rtacha absolyut xatolik): {mae:.2f}")

# Model ma'nosi
print(f"\n💡 AMALIY MA'NO:")
if best_feature == 'advertising_budget':
    print(f"   • Har ${slope:.0f} reklama byudjetiga {slope:.0f} birlik sotuv o'sishi")
elif best_feature == 'temperature':
    print(f"   • Har 1°C harorat o'zgarishiga {slope:.0f} birlik sotuv o'zgarishi")
else:
    print(f"   • Har bir {best_feature} o'zgarishiga {slope:.2f} birlik sotuv o'zgarishi")

print(f"   • Model {r2*100:.0f}% dispersiyani tushuntiradi")

### Vazifa 3.3: Model vizualizatsiyasi

In [ ]:
# VAZIFA: 3 ta grafik yarating:
# 1. Regressiya chizig'i
# 2. Qoldiqlar grafigi
# 3. Haqiqiy vs Bashorat

plt.figure(figsize=(18, 5))

# 1. Regressiya chizig'i
plt.subplot(1, 3, 1)
# Sizning kodingiz:
plt.scatter(X, y, alpha=0.7, s=50, label='Ma\'lumotlar')
plt.plot(X, y_pred, color='red', linewidth=2, label='Regressiya chizig\'i')
plt.xlabel(best_feature.replace('_', ' ').title())
plt.ylabel('Sotuv hajmi')
plt.title(f'Chiziqli regressiya\nR² = {r2:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Qoldiqlar grafigi
plt.subplot(1, 3, 2)
# Sizning kodingiz:
residuals = y - y_pred
plt.scatter(y_pred, residuals, alpha=0.7, s=50)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Bashorat qilingan qiymatlar')
plt.ylabel('Qoldiqlar')
plt.title('Qoldiqlar grafigi')
plt.grid(True, alpha=0.3)

# 3. Haqiqiy vs Bashorat
plt.subplot(1, 3, 3)
# Sizning kodingiz:
plt.scatter(y, y_pred, alpha=0.7, s=50)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Mukammal bashorat')
plt.xlabel('Haqiqiy qiymatlar')
plt.ylabel('Bashorat qilingan qiymatlar')
plt.title('Haqiqiy vs Bashorat')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🔮 4. Bashorat qilish

### Vazifa 4.1: Yangi qiymatlar uchun bashorat

In [ ]:
# VAZIFA: Yangi qiymatlar uchun sotuv hajmini bashorat qiling

# Test qiymatlari
if best_feature == 'advertising_budget':
    test_values = [6000, 8000, 10000, 12000, 15000]
    unit = "$"
elif best_feature == 'temperature':
    test_values = [20, 25, 30, 35, 40]
    unit = "°C"
else:
    # Boshqa xususiyatlar uchun
    min_val = sales_data[best_feature].min()
    max_val = sales_data[best_feature].max()
    test_values = np.linspace(min_val, max_val, 5)
    unit = ""

print(f"🔮 BASHORAT QILISH ({best_feature}):")
print("=" * 40)

for value in test_values:
    # Sizning kodingiz:
    predicted_sales = model.predict([[value]])[0]
    print(f"   📊 {best_feature}: {value}{unit} → Sotuv: {predicted_sales:,.0f}")

# Confidence interval hisoblash (qo'shimcha)
print(f"\n⚠️  OGOHLANTIRISHLAR:")
print(f"   • Ma'lumotlar oralig'i: {sales_data[best_feature].min():.0f} - {sales_data[best_feature].max():.0f}")
print(f"   • Bu oraliqdan tashqarida bashorat xavfli!")
print(f"   • Model xatoligi: ±{rmse:.0f} birlik")

### Vazifa 4.2: Interaktiv bashorat kalkulyatori

In [ ]:
# Bashorat kalkulyatori
def predict_sales(input_value):
    """Berilgan qiymat uchun sotuv hajmini bashorat qilish"""
    predicted = model.predict([[input_value]])[0]
    
    # Ishonch oralig'i (soddalashtirilgan)
    confidence_interval = 1.96 * rmse  # 95% ishonch oralig'i
    lower_bound = predicted - confidence_interval
    upper_bound = predicted + confidence_interval
    
    return predicted, lower_bound, upper_bound

# VAZIFA: Bir necha qiymat uchun bashorat qiling va natijalarni taqqoslang
print(f"🎯 DETALLI BASHORAT NATIJALARI:")
print("=" * 50)

# Sizning kodingiz:
test_input = [6000, 10000, 14000] if best_feature == 'advertising_budget' else [25, 30, 35]

for value in test_input:
    pred, lower, upper = predict_sales(value)
    print(f"\n📊 {best_feature}: {value}")
    print(f"   🎯 Bashorat: {pred:,.0f}")
    print(f"   📊 95% ishonch oralig'i: {lower:,.0f} - {upper:,.0f}")
    
    # Ma'lumotlar oralig'ini tekshirish
    min_range = sales_data[best_feature].min()
    max_range = sales_data[best_feature].max()
    
    if value < min_range or value > max_range:
        print(f"   ⚠️  OGOHLANTIRISH: Ma'lumotlar oralig'idan tashqarida!")
    else:
        print(f"   ✅ Ma'lumotlar oralig'ida")

## 📋 5. Xulosa va tahlil

### Vazifa 5.1: Natijalarni umumlashtirish

In [ ]:
# VAZIFA: Umumiy xulosa chiqaring

print("📋 AMALIY MASHG'ULOT XULOSASI")
print("=" * 50)

print("🎯 ASOSIY NATIJALAR:")
print(f"\n📊 Ma'lumotlar:")
print(f"   • Jami yozuvlar: {len(sales_data)}")
print(f"   • O'zgaruvchilar soni: {len(sales_data.columns)}")
print(f"   • Bog'liq o'zgaruvchi: sotuv hajmi")

print(f"\n🔗 Korrelyatsiya:")
print(f"   • Eng kuchli bog'lanish: {best_feature} (r = {sales_corrs.iloc[0]:.3f})")
print(f"   • Bog'lanish turi: {'Ijobiy' if sales_corrs.iloc[0] > 0 else 'Salbiy'}")
print(f"   • Bog'lanish kuchi: {'Kuchli' if abs(sales_corrs.iloc[0]) > 0.7 else 'O\'rtacha' if abs(sales_corrs.iloc[0]) > 0.3 else 'Zaif'}")

print(f"\n📈 Regressiya modeli:")
print(f"   • R² = {r2:.3f} ({r2*100:.1f}% tushuntiriladi)")
print(f"   • RMSE = {rmse:.2f}")
print(f"   • Model sifati: {'Juda yaxshi' if r2 > 0.8 else 'Yaxshi' if r2 > 0.6 else 'O\'rtacha' if r2 > 0.4 else 'Zaif'}")

print(f"\n💡 AMALIY TAVSIYALAR:")
if best_feature == 'advertising_budget':
    print(f"   • Reklama byudjetini oshirish sotuvni oshiradi")
    print(f"   • Har ${slope:.0f} qo'shimcha byudjet {slope:.0f} birlik ko'p sotuv beradi")
elif best_feature == 'temperature':
    print(f"   • Harorat sotuv hajmiga ta'sir qiladi")
    print(f"   • Optimal harorat diapazoni mavjud bo'lishi mumkin")
else:
    print(f"   • {best_feature} sotuv hajmiga muhim ta'sir ko'rsatadi")

print(f"\n⚠️  CHEKLOVLAR:")
print(f"   • Model faqat {best_feature} asosida bashorat qiladi")
print(f"   • Boshqa omillar hisobga olinmagan")
print(f"   • Ma'lumotlar oralig'idan tashqarida bashorat xavfli")
print(f"   • Korrelyatsiya sabab-oqibat munosabatini bildirmaydi")

## 🎯 Qo'shimcha vazifalar (ixtiyoriy)

### Vazifa 6: Boshqa xususiyatlar bilan tajriba
1. `temperature` o'zgaruvchisi bilan regressiya modeli yarating
2. `promotion_days` bilan korrelyatsiyani tekshiring
3. Qaysi model eng yaxshi natija beradi?

### Vazifa 7: Ma'lumotlarni boyitish
1. Yangi xususiyat yarating: `budget_per_sale = advertising_budget / sales_volume`
2. Bu yangi xususiyat bilan qanday korrelyatsiya bor?
3. Seasonal pattern bormi? (oylar bo'yicha tahlil)

### Vazifa 8: Model yaxshilash
1. Outlier'larni aniqlang va olib tashlang
2. Model sifati yaxshiladimi?
3. Log transformatsiya qo'llang va natijani taqqoslang